# Experiment 01: Baseline Seq2Seq LSTM

This notebook walks through training a baseline Seq2Seq LSTM model step by step.
It is meant to be **read alongside the source code** in `lalango/models/seq2seq_lstm.py`.

By the end of this notebook you will:
- Understand how the encoder and decoder interact
- See how the loss decreases over training
- Generate some example translations

---

**Prerequisite:** Complete Phase 1 — implement `Encoder.forward()`, `Decoder.forward_step()`,
and `Seq2SeqLSTM.forward()` in `lalango/models/seq2seq_lstm.py`.

## Step 1: Prepare a tiny dataset

We use a tiny synthetic dataset here so you can run this notebook quickly
without needing real parallel data.

Replace this with your own data once you have it.

In [ ]:
import sys
sys.path.insert(0, '..')  # make sure we can import from the lalango package

# A tiny toy dataset: English → Pig Latin
# (We use Pig Latin so you can verify the translations are correct by eye)
train_src = [
    "hello", "world", "how are you", "good morning",
    "thank you", "see you later", "yes", "no",
    "please", "sorry",
]
train_tgt = [
    "ellohay", "orldway", "owhay areway ouyay", "oodgay orningmay",
    "anktay ouyay", "eesay ouyay aterlay", "esyay", "onay",
    "easeplay", "orrysay",
]

print(f"Dataset size: {len(train_src)} pairs")
for src, tgt in zip(train_src[:3], train_tgt[:3]):
    print(f"  '{src}' → '{tgt}'")

## Step 2: Build the vocabulary

In [ ]:
from lalango.tokenizers.char_tokenizer import CharTokenizer

src_tokenizer = CharTokenizer()
src_tokenizer.build(train_src)

tgt_tokenizer = CharTokenizer()
tgt_tokenizer.build(train_tgt)

print(f"Source vocab size: {src_tokenizer.vocab_size}")
print(f"Target vocab size: {tgt_tokenizer.vocab_size}")
print()
print("Example encoding:")
encoded = src_tokenizer.encode("hello", add_eos=True)
decoded = src_tokenizer.decode(encoded)
print(f"  'hello' → {encoded} → '{decoded}'")

## Step 3: Encode the dataset and create batches

In [ ]:
from lalango.data.dataset import encode_corpus, create_batches

encoded_pairs = encode_corpus(train_src, train_tgt, src_tokenizer)
batches = create_batches(encoded_pairs, batch_size=4)

print(f"Total batches: {len(batches)}")
print(f"First batch source shape: {len(batches[0]['source'])} x {len(batches[0]['source'][0])}")
print(f"First batch target shape: {len(batches[0]['target'])} x {len(batches[0]['target'][0])}")

## Step 4: Create the model

⚠️ This cell requires `Seq2SeqLSTM` to be implemented in `lalango/models/seq2seq_lstm.py`.
If you have not done that yet, this will raise a `NotImplementedError`.

In [ ]:
import torch
from lalango.models.seq2seq_lstm import Seq2SeqLSTM

model = Seq2SeqLSTM(
    src_vocab_size=src_tokenizer.vocab_size,
    tgt_vocab_size=tgt_tokenizer.vocab_size,
    embed_dim=32,    # small values for this toy experiment
    hidden_dim=64,
)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model created with {total_params:,} trainable parameters")

## Step 5: Train for a few epochs

In [ ]:
import torch.nn as nn

optimizer = torch.optim.Adam(model.parameters(), lr=0.005)
criterion = nn.CrossEntropyLoss(ignore_index=0)  # ignore PAD tokens

losses = []

for epoch in range(1, 51):  # 50 epochs on this tiny dataset
    model.train()
    epoch_loss = 0.0

    for batch in batches:
        source = torch.tensor(batch["source"])
        target = torch.tensor(batch["target"])

        optimizer.zero_grad()
        predictions = model(source, target)

        # predictions: [batch, tgt_len, vocab] → [batch*(tgt_len-1), vocab]
        # targets:     [batch, tgt_len]        → [batch*(tgt_len-1)]
        preds  = predictions[:, 1:, :].reshape(-1, tgt_tokenizer.vocab_size)
        labels = target[:, 1:].reshape(-1)

        loss = criterion(preds, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(batches)
    losses.append(avg_loss)

    if epoch % 10 == 0:
        print(f"Epoch {epoch:3d} | Loss: {avg_loss:.4f}")

print("\nTraining complete!")

## Step 6: Plot the training loss

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.plot(losses)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Loss — Seq2Seq LSTM")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Starting loss: {losses[0]:.4f}")
print(f"Final loss:    {losses[-1]:.4f}")

## Step 7: Generate some translations

In [ ]:
model.eval()

test_sentences = ["hello", "world", "yes", "no"]

print("Translations after training:\n")
for sentence in test_sentences:
    encoded = src_tokenizer.encode(sentence, add_eos=True)
    source_tensor = torch.tensor([encoded])
    
    with torch.no_grad():
        output_indices = model.translate(source_tensor)
    
    translation = tgt_tokenizer.decode(output_indices)
    print(f"  '{sentence}' → '{translation}'")

---

## What to try next

- Try increasing `embed_dim` and `hidden_dim` — does the loss go lower?
- Try training for more epochs — when does the model start to overfit?
- Replace the toy dataset with a real parallel corpus
- Move on to **Phase 2** and add Bahdanau Attention — does it help?